# 第四輪實驗候選：快速重現CSV／可選完整重訓
主檔尚未證實Kaggle0.680。預設使用已保存機率，不重訓；兩份CSV已驗證逐byte可重現。若要完整重訓，將下方FULL_RETRAIN改True。
先從GitHub研究分支Code → Download ZIP下載完整儲存庫並上傳；若用第四輪精簡研究包，另附train.csv、test.csv、sample_submission.csv。私人repo請先在瀏覽器登入下載，不需要Token。


In [ ]:
from pathlib import Path
import os,sys,subprocess,json,zipfile,time,shutil
base=Path.cwd()
matches=[p.parent for p in base.rglob('train.csv') if (p.parent/'research_0680'/'src').is_dir() and '.venvs' not in p.parts]
if not matches:
    try:
        from google.colab import files
    except ImportError:
        raise RuntimeError('請在既有儲存庫目錄執行，或在 Colab 上傳完整儲存庫 ZIP。')
    uploaded=files.upload()
    target=base/'earthquake_repository';target.mkdir(exist_ok=True)
    for name in uploaded:
        if name.endswith('.zip'):
            with zipfile.ZipFile(name) as archive:
                for item in archive.infolist():
                    destination=(target/item.filename).resolve()
                    if not destination.is_relative_to(target.resolve()):
                        raise ValueError('拒絕 ZIP 路徑越界')
                archive.extractall(target)
    for project in target.rglob('research_0680'):
        if not (project/'src').is_dir():continue
        for name in ['train.csv','test.csv','sample_submission.csv']:
            if name in uploaded and not (project.parent/name).exists():
                shutil.copyfile(base/name,project.parent/name)
    matches=[p.parent for p in target.rglob('train.csv') if (p.parent/'research_0680'/'src').is_dir()]
if not matches:
    raise RuntimeError('找不到同時含 train.csv 與 research_0680/src 的完整儲存庫。請使用 GitHub Code → Download ZIP。')
repo=matches[0].resolve()
assert all((repo/f).is_file() for f in ['train.csv','test.csv','sample_submission.csv'])
print('Repository:',repo)


In [ ]:
subprocess.check_call([sys.executable,'-m','pip','install','-r',str(repo/'research_0680/configs/requirements.txt')])
for name in ['OPENBLAS_NUM_THREADS','OMP_NUM_THREADS','MKL_NUM_THREADS']:
    os.environ[name]='1'
run_dir=repo/'research_0680'/'phase4_colab_runs'/time.strftime('%Y%m%d_%H%M%S')
run_dir.mkdir(parents=True,exist_ok=False)
os.environ['MPLCONFIGDIR']=str(run_dir/'matplotlib_cache')
os.environ['RESEARCH0680_PHASE4_POOL_RESULTS']=str(run_dir/'pooling_results')
FULL_RETRAIN=False
print('Output:',run_dir,'Full retraining:',FULL_RETRAIN)


## 執行
預設只合併包內來源機率與生成CSV，不訓練。FULL_RETRAIN=True會在新目錄重訓三個主候選來源規則，CPU可能需要十多分鐘。每折TE與模型範圍保持一致，不把fold模型的TE偷換成全資料TE。所有CSV為實驗檔，不能將本地普通Accuracy當成Kaggle實測成績。


In [ ]:
if FULL_RETRAIN:
    for envname,subdir in [('RESEARCH0680_PHASE3_RESULTS','compact'),('RESEARCH0680_PHASE4_RESULTS','expanded'),('RESEARCH0680_PHASE4_PAIR_RESULTS','pairwise')]:
        os.environ[envname]=str(run_dir/subdir)
    for module in ['research_0680.phase3.run','research_0680.phase4.run','research_0680.phase4.pairwise']:
        subprocess.check_call([sys.executable,'-B','-m',module],cwd=repo,env=os.environ.copy())
subprocess.check_call([sys.executable,'-B','-m','research_0680.phase4.pool'],cwd=repo,env=os.environ.copy())


In [ ]:
import hashlib
for name in ['submission_candidate_phase4_pool_argmax.csv','submission_candidate_phase4_pool_balanced1700.csv']:
    path=run_dir/'pooling_results'/name
    print(name,path.stat().st_size,hashlib.sha256(path.read_bytes()).hexdigest())
print((run_dir/'pooling_results/oof_comparison.csv').read_text())
print('Kaggle score unknown; statistical improvement unconfirmed.')


## 下載
argmax為主檔；balanced1700依賴測試各1700的額外假設。本輪未證實超過0.680，不替換既有0.679。


In [ ]:
bundle=base/'Kaggle_Phase4_reproduction_results.zip'
with zipfile.ZipFile(bundle,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in run_dir.rglob('*'):
        if p.is_file() and 'matplotlib_cache' not in p.parts:
            z.write(p,p.relative_to(run_dir))
print(bundle,bundle.stat().st_size)
try:
    from google.colab import files
    files.download(str(bundle))
except ImportError:
    print('本機結果位於上面列出的 ZIP 路徑。')
